# 11 — Best Practices

Notebook terakhir dari seri fundamentals. Ini tentang kebiasaan yang bikin notebook lo dari 'jalan di komputer gue' jadi 'reproducible, maintainable, publishable'.

**Yang akan lo pelajari:**
- Struktur notebook yang baik
- Reproducibility (random seed, fixed environment)
- Dokumentasi yang efektif
- Manajemen output
- Export notebook
- Checklist sebelum publish

---

## 1. Struktur Notebook yang Baik

### Template struktur yang recommended:

```
# Judul Notebook
Deskripsi singkat: apa yang dilakukan notebook ini, tujuannya apa.

## Setup
- Import semua library
- Set constants
- Set random seed

## 1. Load Data
- Baca data
- Quick preview

## 2. Exploratory Data Analysis (EDA)
- Shape, dtypes, missing values
- Distribusi
- Korelasi

## 3. Preprocessing
- Handle missing values
- Feature engineering
- Train-test split

## 4. Modeling
- Definisi model
- Training
- Evaluasi

## 5. Hasil dan Kesimpulan
- Summary hasil
- Next steps
```

## 2. Setup Cell — Selalu di Awal

In [ ]:
# ============================================================
# SETUP — Selalu cell pertama
# ============================================================

# Standard Library
import os
import sys
import json
import random
from pathlib import Path
from datetime import datetime

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Jupyter
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# ============================================================
# CONSTANTS
# ============================================================
RANDOM_SEED = 42
DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')
MODEL_DIR = Path('models')

# ============================================================
# REPRODUCIBILITY
# ============================================================
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# DISPLAY SETTINGS
# ============================================================
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

print(f"Setup selesai — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")

## 3. Reproducibility

In [ ]:
# Reproducibility = hasil yang sama setiap dijalankan ulang

# TANPA seed — hasil berbeda setiap run
print("Tanpa seed:")
print(np.random.randint(0, 100, 5))
print(np.random.randint(0, 100, 5))  # beda!

print()

# DENGAN seed — hasil selalu sama
print("Dengan seed 42:")
np.random.seed(42)
print(np.random.randint(0, 100, 5))

np.random.seed(42)  # reset seed
print(np.random.randint(0, 100, 5))  # sama!

In [ ]:
# Untuk ML: set seed di semua library yang dipakai
SEED = 42

random.seed(SEED)          # Python stdlib
np.random.seed(SEED)       # NumPy
os.environ['PYTHONHASHSEED'] = str(SEED)  # Python hash

# Kalau pakai TensorFlow:
# import tensorflow as tf
# tf.random.set_seed(SEED)

# Kalau pakai PyTorch:
# import torch
# torch.manual_seed(SEED)
# torch.backends.cudnn.deterministic = True

print(f"Semua random seed di-set ke {SEED}")

In [ ]:
# Catat versi environment untuk reproducibility
def print_environment_info():
    """Print info environment yang relevan untuk reproducibility."""
    print("=" * 50)
    print("ENVIRONMENT INFO")
    print("=" * 50)
    print(f"Date      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Python    : {sys.version.split()[0]}")
    print(f"Platform  : {sys.platform}")
    
    libs = ['numpy', 'pandas', 'matplotlib']
    for lib in libs:
        try:
            mod = __import__(lib)
            print(f"{lib:<12}: {mod.__version__}")
        except ImportError:
            print(f"{lib:<12}: not installed")
    print("=" * 50)

print_environment_info()

## 4. Dokumentasi yang Efektif

### Tips dokumentasi di Jupyter:

**1. Satu cell = satu ide**
Jangan buat cell yang terlalu panjang. Kalau cell lo scroll panjang, pecah jadi beberapa.

**2. Markdown sebelum code kompleks**
```
## Normalisasi Fitur
Kita pakai Z-score normalization supaya setiap fitur punya skala sama.
Ini mencegah fitur dengan nilai besar mendominasi model.
```

**3. Jelaskan "kenapa", bukan "apa"**
```python
# ❌ Kurang berguna
# Hitung mean
mean = df['nilai'].mean()

# ✅ Lebih berguna  
# Kita pakai mean sebagai nilai default untuk missing values
# karena distribusi nilai cukup normal (tidak ada outlier ekstrem)
mean = df['nilai'].mean()
```

**4. Kesimpulan per section**
Di akhir setiap section besar, tulis 1-2 kalimat kesimpulan temuan.

In [ ]:
# Docstring yang baik untuk function
def bersihkan_data(df, kolom_target, strategi='mean'):
    """
    Bersihkan missing values di DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame input
    kolom_target : str atau list
        Nama kolom yang akan dibersihkan
    strategi : {'mean', 'median', 'drop'}, default 'mean'
        Strategi handling missing values:
        - 'mean': isi dengan nilai rata-rata
        - 'median': isi dengan nilai tengah
        - 'drop': hapus baris yang ada missing values
    
    Returns
    -------
    pd.DataFrame
        DataFrame yang sudah dibersihkan
    
    Raises
    ------
    ValueError
        Kalau strategi tidak valid
    
    Examples
    --------
    >>> df = pd.DataFrame({'nilai': [85, None, 92]})
    >>> bersihkan_data(df, 'nilai', 'mean')
    """
    import numpy as np
    import pandas as pd
    
    if isinstance(kolom_target, str):
        kolom_target = [kolom_target]
    
    df_clean = df.copy()
    
    for kolom in kolom_target:
        if strategi == 'mean':
            df_clean[kolom] = df_clean[kolom].fillna(df_clean[kolom].mean())
        elif strategi == 'median':
            df_clean[kolom] = df_clean[kolom].fillna(df_clean[kolom].median())
        elif strategi == 'drop':
            df_clean = df_clean.dropna(subset=[kolom])
        else:
            raise ValueError(f"Strategi '{strategi}' tidak valid. Pilih: mean, median, drop")
    
    return df_clean

# Test
df_test = pd.DataFrame({'nilai': [85, None, 92, None, 78], 'nama': ['A','B','C','D','E']})
print(bersihkan_data(df_test, 'nilai', 'mean'))

## 5. Manajemen Output dan Cell

In [ ]:
# Clear output sebelum commit ke git
# Caranya: Kernel → Restart & Clear Output
# Atau: Edit → Clear All Outputs

# Kenapa? Output bisa:
# 1. Bikin file .ipynb jadi besar (gambar, data besar)
# 2. Menyimpan data sensitif
# 3. Bikin diff git jadi berantakan

# Tips: sebelum push ke GitHub, selalu:
# Kernel → Restart & Run All
# Verifikasi semua cell jalan dari atas ke bawah

print("Tips manajemen output:")
tips = [
    "Run All sebelum push — pastikan bisa dijalankan ulang",
    "Clear output sebelum commit kalau ada data sensitif",
    "Simpan gambar penting ke file, jangan cuma output cell",
    "Pakai nbstripout untuk otomatis clear output saat git commit",
]
for i, tip in enumerate(tips, 1):
    print(f"  {i}. {tip}")

In [ ]:
# Cegah output yang terlalu panjang
import pandas as pd
import numpy as np

df_besar = pd.DataFrame(np.random.randn(1000, 10))

# ❌ Jangan
# df_besar  # ini print 1000 baris!

# ✅ Preview yang controlled
print(f"Shape: {df_besar.shape}")
display(df_besar.head())
print("...")
display(df_besar.tail(3))

## 6. Export Notebook

In [ ]:
# Export notebook ke berbagai format via command line
# (jalankan di terminal, bukan di notebook)

export_commands = {
    "HTML (share di web)": "jupyter nbconvert --to html notebook.ipynb",
    "PDF (laporan)": "jupyter nbconvert --to pdf notebook.ipynb",
    "Python script": "jupyter nbconvert --to script notebook.ipynb",
    "Markdown": "jupyter nbconvert --to markdown notebook.ipynb",
    "Slides": "jupyter nbconvert --to slides notebook.ipynb",
}

print("Commands export notebook:")
for format_name, cmd in export_commands.items():
    print(f"\n{format_name}:")
    print(f"  $ {cmd}")

In [ ]:
# Export dari dalam notebook
import subprocess

# Ini akan export notebook yang SEDANG DIJALANKAN
# ke HTML di direktori yang sama

# notebook_name = '11_best_practices.ipynb'
# result = subprocess.run(
#     ['jupyter', 'nbconvert', '--to', 'html', notebook_name],
#     capture_output=True, text=True
# )
# print(result.stdout)
# print(result.stderr)

print("Uncomment kode di atas untuk export ke HTML")

## 7. Checklist Final

In [ ]:
# Checklist sebelum publish/share notebook

checklist = {
    "REPRODUCIBILITY": [
        "Random seed di-set di awal",
        "Bisa di-run ulang dari atas ke bawah tanpa error (Restart & Run All)",
        "Versi library dicatat (pip freeze atau watermark)",
        "Data source dicatat (URL, tanggal download, versi dataset)",
    ],
    "STRUKTUR": [
        "Judul dan deskripsi di awal",
        "Import semua di cell pertama",
        "Heading yang jelas per section",
        "Kesimpulan / next steps di akhir",
    ],
    "CODE QUALITY": [
        "Tidak ada cell yang jalan dalam urutan yang aneh",
        "Variabel punya nama yang meaningful",
        "Function kompleks ada docstring",
        "Magic numbers di-extract jadi konstanta",
    ],
    "OUTPUT": [
        "Output cell tidak menyimpan data sensitif",
        "Plot punya label axis dan judul",
        "Output yang panjang di-limit (head/tail)",
    ],
}

for kategori, items in checklist.items():
    print(f"\n{'='*40}")
    print(f"  {kategori}")
    print(f"{'='*40}")
    for item in items:
        print(f"  ☐ {item}")

In [ ]:
# Watermark — catat info environment otomatis
# pip install watermark
try:
    %load_ext watermark
    %watermark -v -p numpy,pandas,matplotlib
except:
    # Manual fallback kalau watermark tidak ada
    import sys, numpy as np, pandas as pd, matplotlib
    print(f"Python    : {sys.version.split()[0]}")
    print(f"NumPy     : {np.__version__}")
    print(f"Pandas    : {pd.__version__}")
    print(f"Matplotlib: {matplotlib.__version__}")

---

## 🎉 Selamat!

Lo udah selesaikan semua 11 notebook fundamentals. Sekarang lo punya fondasi yang solid untuk lanjut ke:

### Next Learning Path:
- 📁 `machine-learning/` — dataset preparation, model training, evaluation
- 📁 `computer-vision/` — image processing, object detection
- 📁 `ai-modern/` — embeddings, RAG, prompt engineering

### Yang sudah lo kuasai:
- ✅ Jupyter interface, shortcuts, dan kernel
- ✅ Python fundamentals untuk data science
- ✅ Import dan manajemen package
- ✅ Magic commands untuk produktivitas
- ✅ File I/O (teks, CSV, JSON, Excel)
- ✅ NumPy — array, indexing, broadcasting, math
- ✅ Pandas — DataFrame, cleaning, groupby, merge
- ✅ Matplotlib — berbagai tipe plot
- ✅ ipywidgets — notebook interaktif
- ✅ Debugging dan profiling
- ✅ Best practices dan reproducibility

---

> *"The best way to learn is to build something."*
> Sekarang coba terapkan semua ini ke proyek nyata!